# Document Pre-processing for Knowledge Tuning

## Overview

This notebook demonstrates a complete document preprocessing pipeline designed specifically for **knowledge tuning** with sdg-hub. 

## What This Notebook Does

This preprocessing pipeline transforms raw documents (PDFs, Word docs, etc.) into seed data for data generation:

1. **Document Parsing**: Converts raw documents to structured markdown format
2. **Chunking**: Splits documents into manageable chunks while preserving structure and context
3. **Seed Data Creation**: Formats chunks with in-context learning (ICL) templates for effective knowledge tuning

## Prerequisites

- We will use the existing InstructLab document parser (`docparser_v2.py`) and Document parsing configuration (`docling_v2_config.yaml`)
- Raw pdf documents in the `document_collection` directory


In [ ]:
# Step 1: Install Required Dependencies
# Install packages needed for document processing

%pip install --quiet docling ipywidgets

In [ ]:
# Step 2: Document Processing Pipeline
# Define the directory containing raw documents to be processed
data_dir = "document_collection"

# Run the document parser to convert documents to markdown
# - input-dir: Directory containing source documents
# - output-dir: Directory where processed markdown files will be saved
# - c: Configuration file specifying parsing parameters
!python ../../docparser_v2.py --input-dir {data_dir} --output-dir {data_dir} -c ../../docling_v2_config.yaml

In [ ]:
# Step 3: List Processed Documents
import glob

# In our example above docling step produces markdown of all the pdf files in the document_collection
list_md_files = glob.glob(f"{data_dir}/*.md")

In [ ]:
# Step 4: Text Chunking and Dataset Creation
import sys
import os

sys.path.insert(0, os.path.dirname(os.path.dirname(os.getcwd())))
from knowledge_utils import DocProcessor

# Prepare seed data for the SDG-Hub knowledge pipeline by splitting documents into chunks and combining each chunk with each ICL example from a YAML file
user_config_path = f"{data_dir}/qna.yaml"
dp = DocProcessor(data_dir, user_config_path=user_config_path)
seed_data = dp.get_processed_markdown_dataset(list_md_files)

# Save the seed data to a JSONL file for downstream use
sdg_demo_output = "sdg_demo_output"
seed_data_path = f"{sdg_demo_output}/seed_data.jsonl"
os.makedirs(sdg_demo_output, exist_ok=True)
seed_data.to_json(seed_data_path, orient="records", lines=True, force_ascii=False)

### Next Steps:
- The `{sdg_demo_output}/seed_data.jsonl` file is now ready for the knowledge tuning pipeline.
- You can now refer to the [knowledge generation](knowledge_generation_ja.ipynb) notebook